In [4]:
from main_utils import (
    download_dataset
)
from dotenv import load_dotenv
load_dotenv()

download_dataset()

INFO:MAIN:`data` folder not found. Downloading dataset...


100%|██████████| 1.94G/1.94G [06:10<00:00, 5.63MB/s]

Extracting files...



INFO:MAIN:Dataset downloaded and moved to `data`.


In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchvision.transforms as T
import timm
from transformers import AutoModel
from wildlife_datasets.datasets import AnimalCLEF2026
from wildlife_tools.features import DeepFeatures
from wildlife_tools.similarity import CosineSimilarity

root = 'data'
dataset_full = AnimalCLEF2026(
    root,
    transform=None,
    load_label=True,
    factorize_label=True,
    check_files=False
)
dataset_full = dataset_full.get_subset(dataset_full.df['split'] == 'test')

datasets = {}
for name in dataset_full.metadata['dataset'].unique():
    datasets[name] = dataset_full.get_subset(dataset_full.df['dataset'] == name)

Exception: root does not exist. You may have have mispelled it.

In [ ]:
from wildlife_tools.similarity.wildfusion import SimilarityPipeline, WildFusion
from transformers.dynamic_module_utils import get_class_from_dynamic_module

device = 'cuda'
batch_size = 4

pipeline_mega = SimilarityPipeline(
    matcher = CosineSimilarity(),
    extractor = DeepFeatures(
        model = timm.create_model("hf-hub:BVRA/MegaDescriptor-L-384", pretrained=True).eval(),
        device=device,
        batch_size=batch_size,
        cache_path="features/mega"
    ),
    transform = T.Compose([
        T.Resize(size=(384, 384)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]),
    calibration = None
)


MiewIdNet = get_class_from_dynamic_module(
    "modeling_miewid.MiewIdNet",
    "conservationxlabs/miewid-msv3",
)
if not hasattr(MiewIdNet, "all_tied_weights_keys"):
    MiewIdNet.all_tied_weights_keys = {}

pipeline_miew =  SimilarityPipeline(
    matcher = CosineSimilarity(),
    extractor = DeepFeatures(
        model = AutoModel.from_pretrained('conservationxlabs/miewid-msv3', trust_remote_code=True),
        device=device,
        batch_size=batch_size,
        cache_path="features/miew"
    ),
    transform = T.Compose([
        T.Resize(size=(512, 512)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]),
    calibration = None
)

Building Model Backbone for efficientnetv2_rw_m model
config.model_name efficientnetv2_rw_m
model_name efficientnetv2_rw_m
final_in_features 2152


Loading weights: 100%|██████████| 1210/1210 [00:00<00:00, 12352.79it/s]


In [3]:
pipelines = [pipeline_mega, pipeline_miew]

similarities = {}
for name, dataset in datasets.items():
    B = len(dataset)
    wildfusion = WildFusion(calibrated_pipelines = pipelines, priority_pipeline = pipeline_mega)
    similarity = wildfusion(dataset, dataset, B=B)    
    similarities[name] = similarity.astype(np.float64)

100%|███████████████████████████████████████████████████████████████| 69/69 [00:09<00:00,  7.29it/s]


In [ ]:
from sklearn.cluster import HDBSCAN

def run_HDBSCAN(similarity):
    # Convert similarity (high is good) to distance (small is good)
    distance = (np.max(similarity) - np.maximum(similarity, 0)) / np.max(similarity)
    # Obtain predictions
    clustering = HDBSCAN(min_cluster_size=2, metric='precomputed')
    clusters = clustering.fit(distance)
    # Relabel -1 clusters into separate clusters
    labels = np.array(clusters.labels_)
    neg_indices = np.where(labels == -1)[0]
    new_labels = np.arange(labels.max()+1, labels.max()+1+len(neg_indices))
    labels[neg_indices] = new_labels
    return labels

In [5]:
results = pd.DataFrame()
for name, similarity in similarities.items():
    # Save the clusters for one dataset
    clusters = run_HDBSCAN(similarity)
    result = pd.DataFrame({
        'image_id': datasets[name].metadata['image_id'],
        'cluster': [f'cluster_{name}_{c}' for c in clusters]
    })
    # Merge the clusters to other datasets
    results = pd.concat((results, result))
results = results.reset_index(drop=True)
results.to_csv('submission.csv', index=False)

## Summary

This technique slightly improves the performance (from 19.4% to 20.3%). However, it removes the need to tune the pesky hyperparameter `eps` manually and is ready for additional generalisations such as:

- Replace MegaDescriptor or MiewID with other models, such as the local feature model SuperPoint or any other.
- Replace the uncalibrated models with their calibrated versions. This will be necessary when using a local feature model, which returns the number of matching keypoints, which is a much higher score than the cosine similarity.

Additional improvements may be achieved by:

- Use automatic tools for data preprocessing, such as image segmentation.